# REPA × ImageNet-256 × DINOv2

Trains **SiT-S/8** on ImageNet-256 using **DINOv2 ViT-Base** as the representation encoder.

**Requirements (Kaggle datasets attached to this notebook):**
- `dimensi0n/imagenet-256` — ImageNet 1K resized to 256x256

All code runs from the cloned `nikiboura/REPA` fork.

## 1. Install dependencies

In [ ]:
%%bash
pip install -q accelerate diffusers timm einops

## 2. Clone REPA fork

In [ ]:
%%bash
cd /kaggle/working
# check if folder exists already
if [ ! -d "REPA" ]; then
    git clone https://github.com/nikiboura/REPA.git REPA
fi
# latest git commit
echo "REPA ready. Commit: $(git -C REPA rev-parse --short HEAD)"

## 3. Set paths

In [ ]:
import os

REPA_DIR      = '/kaggle/working/REPA'
DATA_DIR      = '/kaggle/working/data/imagenet256'
IMAGENET_ROOT = '/kaggle/input/imagenet-256'

os.makedirs(DATA_DIR, exist_ok=True)

print(f'REPA dir     : {REPA_DIR}')
print(f'Data dir     : {DATA_DIR}')
print(f'ImageNet dir : {os.path.isdir(IMAGENET_ROOT)}')

## 4. Prepare ImageNet-256
Reads ImageNet-256 images (already 256x256), VAE-encodes each image, saves latents as `.npy`.

In [ ]:
import subprocess, sys, threading

cmd = [
    sys.executable, '/kaggle/working/REPA/prepare_imagenet256.py',
    '--out-dir',           '/kaggle/working/data/imagenet256',
    '--imagenet-root',     '/kaggle/input/imagenet-256',
    '--resolution',        '256',
    '--max-train-samples', '5000',
    '--max-val-samples',   '5000',
]
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

def stream(pipe):
    for line in pipe:
        print(line, end='', flush=True)

t1 = threading.Thread(target=stream, args=(process.stdout,))
t2 = threading.Thread(target=stream, args=(process.stderr,))
t1.start(); t2.start()
t1.join(); t2.join()
process.wait()
print('Exit code:', process.returncode)

## 5. Train
Runs 15000 steps. Checkpoints saved to `/kaggle/working/results/`.

**Models**
1. SiT-S/8 (Diffusion Model): denoises in latent space
2. DINOv2 ViT-Base: pretrained vision encoder (downloads automatically)

**Losses**:
1. denoising_loss: standard diffusion loss
2. proj_loss: REPA alignment loss

Total loss = denoising_loss + proj_loss * proj_coeff

In [ ]:
import subprocess, os, re
from tqdm.notebook import tqdm

env = os.environ.copy()

cmd = [
    'accelerate', 'launch',
    '--mixed_precision', 'fp16',
    '--num_processes', '1',
    '/kaggle/working/REPA/train.py',
    '--exp-name',            'repa-sits8-dinov2-imagenet256',
    '--output-dir',          '/kaggle/working/results',
    '--report-to',           'none',
    '--model',               'SiT-S/8',
    '--num-classes',         '1000',
    '--encoder-depth',       '8',
    '--enc-type',            'dinov2-vit-b',
    '--data-dir',            '/kaggle/working/data/imagenet256',
    '--resolution',          '256',
    '--batch-size',          '32',
    '--num-workers',         '2',
    '--epochs',              '200',
    '--learning-rate',       '1e-4',
    '--mixed-precision',     'fp16',
    '--proj-coeff',          '0.5',
    '--cfg-prob',            '0.1',
    '--path-type',           'linear',
    '--max-train-steps',     '15000',
    '--checkpointing-steps', '15000',
    '--sampling-steps',      '99999999',
]

process = subprocess.Popen(
    cmd, cwd='/kaggle/working/REPA', env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

pat = r'Steps:\s*(\d+)/\d+.*?loss=([\d.]+).*?proj_loss=(-?[\d.]+)'
pbar = tqdm(total=15000, desc='Training DINOv2')
last_step = 0
for line in process.stdout:
    m = re.search(pat, line)
    if m:
        step = int(m.group(1))
        if step > last_step:
            pbar.update(step - last_step)
            pbar.set_postfix({'loss': m.group(2), 'proj': m.group(3)})
            last_step = step
    else:
        stripped = line.strip()
        if stripped:
            print(stripped, flush=True)
pbar.close()
process.wait()
print('Training complete. Checkpoint saved.')

## 6. Generate samples

In [ ]:
import subprocess, os, glob

env = os.environ.copy()

ckpts = sorted(glob.glob('/kaggle/working/results/repa-sits8-dinov2-imagenet256/checkpoints/*.pt'))
print('Using checkpoint:', ckpts[-1])

cmd = [
    'torchrun', '--nproc_per_node=1',
    '/kaggle/working/REPA/generate.py',
    '--model',                'SiT-S/8',
    '--ckpt',                 ckpts[-1],
    '--encoder-depth',        '8',
    '--num-classes',          '1000',
    '--projector-embed-dims', '768',
    '--per-proc-batch-size',  '16',
    '--num-fid-samples',      '2000',
    '--path-type',            'linear',
    '--mode',                 'ode',
    '--num-steps',            '50',
    '--cfg-scale',            '1.0',
    '--resolution',           '256',
    '--vae',                  'mse',
]

result = subprocess.run(cmd, cwd='/kaggle/working/REPA', env=env, text=True)
print('Exit code:', result.returncode)

## 7. Show generated images

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import glob

npz_files = sorted(glob.glob('/kaggle/working/REPA/samples/**/*.npz', recursive=True))

# loads images
data = np.load(npz_files[-1])
imgs = data[data.files[0]]  # shape (64, 256, 256, 3)

fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for i, ax in enumerate(axes.flat):
    ax.imshow(imgs[i].astype('uint8'))
    ax.axis('off')
plt.tight_layout()
plt.show()
print(f'From: {npz_files[-1]}')

## 8. Compute FID score

In [ ]:
import subprocess, glob, numpy as np, os
from PIL import Image
from tqdm import tqdm

subprocess.run(['pip', 'install', '-q', 'torch-fidelity'], check=True)

gen_npz = sorted(glob.glob('/kaggle/working/REPA/samples/**/*.npz', recursive=True))[-1]
print('Generated:', gen_npz)

# save generated images to folder
gen_dir = '/kaggle/working/gen_images'
os.makedirs(gen_dir, exist_ok=True)
data = np.load(gen_npz)
imgs = data[data.files[0]]
for i, img in enumerate(tqdm(imgs, desc='Saving generated images')):
    Image.fromarray(img.astype('uint8')).save(f'{gen_dir}/{i:05d}.png')

real_dir = '/kaggle/working/data/imagenet256/images/val'

subprocess.run([
    'fidelity',
    '--gpu', '0',
    '--fid',
    '--input1', gen_dir,
    '--input2', real_dir,
], text=True)